In [14]:
import pandas as pd
import numpy as np

RAW = "Data/raw"
PROCESSED = "Data/processed"

In [15]:
def clean_cpcb(file_path, station_name):

    df = pd.read_csv(file_path, skiprows=9)

    df.columns = [
        "from_time",
        "to_time",
        "PM2.5",
        "PM10",
        "NO2",
        "NOx",
        "SO2",
        "CO",
        "Ozone"
    ]

    df["from_time"] = pd.to_datetime(
        df["from_time"],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

    df["to_time"] = pd.to_datetime(
        df["to_time"],
        format="%d-%m-%Y %H:%M",
        errors="coerce"
    )

    pollutants = [
        "PM2.5", "PM10", "NO2",
        "NOx", "SO2", "CO", "Ozone"
    ]

    for col in pollutants:
        df[col] = pd.to_numeric(
            df[col], errors="coerce"
        )

    df["station"] = station_name

    df = df.dropna(subset=["from_time"])
    df = df.sort_values("from_time")

    return df.reset_index(drop=True)

In [16]:
dwarka = clean_cpcb(
    f"{RAW}/Dwarka_Diwali.csv",
    "Dwarka"
)

igi = clean_cpcb(
    f"{RAW}/IGI_Diwali.csv",
    "IGI"
)

nsit = clean_cpcb(
    f"{RAW}/NSIT_Diwali.csv",
    "NSIT"
)

print("Dwarka:", dwarka.shape)
print("IGI:", igi.shape)
print("NSIT:", nsit.shape)

Dwarka: (191, 10)
IGI: (191, 10)
NSIT: (191, 10)


In [17]:
imd = pd.read_csv(f"{RAW}/IGI_IMD.csv")
imd["time"] = pd.to_datetime(
    imd["time"],
    errors="coerce"
)
imd = imd.sort_values("time")

print(imd.head())
print(imd.shape)

                 time  temp  dwpt  rhum  prcp  snow  wdir  wspd  wpgt  pres  \
0 2025-10-18 00:30:00    24  18.0    69   0.0   NaN     0     0   NaN  1013   
1 2025-10-18 01:30:00    23  18.1    74   0.0   NaN   258     4   NaN  1013   
2 2025-10-18 02:30:00    23  19.0    78   0.0   NaN   231     4   NaN  1012   
3 2025-10-18 03:30:00    23  19.0    78   0.0   NaN     0     0   NaN  1012   
4 2025-10-18 04:30:00    21  18.9    88   0.0   NaN     0     0   NaN  1012   

   tsun  coco  
0   NaN     5  
1   NaN     5  
2   NaN     5  
3   NaN     5  
4   NaN     5  
(192, 12)


In [18]:
dwarka.to_csv(
    f"{PROCESSED}/Dwarka_Clean.csv",
    index=False
)

igi.to_csv(
    f"{PROCESSED}/IGI_Clean.csv",
    index=False
)

nsit.to_csv(
    f"{PROCESSED}/NSIT_Clean.csv",
    index=False
)

imd.to_csv(
    f"{PROCESSED}/IMD_Clean.csv",
    index=False
)

air_quality = pd.concat(
    [dwarka, igi, nsit],
    ignore_index=True
)

air_quality.to_csv(
    f"{PROCESSED}/AirQuality_AllStations.csv",
    index=False
)

print("All files cleaned and saved!")

All files cleaned and saved!


In [19]:
for name, df in {
    "Dwarka": dwarka,
    "IGI": igi,
    "NSIT": nsit,
    "IMD": imd
}.items():

    print("\n", name)
    print(df.isnull().sum())


 Dwarka
from_time      0
to_time        0
PM2.5         10
PM10          16
NO2            0
NOx          191
SO2            6
CO             0
Ozone        191
station        0
dtype: int64

 IGI
from_time      0
to_time        0
PM2.5          6
PM10           7
NO2            0
NOx          191
SO2          191
CO           191
Ozone        191
station        0
dtype: int64

 NSIT
from_time      0
to_time        0
PM2.5          7
PM10           9
NO2            0
NOx          191
SO2            0
CO             0
Ozone        191
station        0
dtype: int64

 IMD
time      0
temp      0
dwpt      0
rhum      0
prcp      0
snow    192
wdir      0
wspd      0
wpgt    192
pres      0
tsun    192
coco      0
dtype: int64


In [20]:
for name, df in {
    "Dwarka": dwarka,
    "IGI": igi,
    "NSIT": nsit
}.items():

    print(
        name,
        "Duplicate timestamps:",
        df["from_time"].duplicated().sum()
    )

Dwarka Duplicate timestamps: 0
IGI Duplicate timestamps: 0
NSIT Duplicate timestamps: 0


In [21]:
for name, df in {
    "Dwarka": dwarka,
    "IGI": igi,
    "NSIT": nsit
}.items():

    expected = pd.date_range(
        "2025-10-18 00:00",
        "2025-10-25 23:00",
        freq="h"
    )

    missing = expected.difference(df["from_time"])

    print(name, "Missing timestamps:", list(missing))

Dwarka Missing timestamps: [Timestamp('2025-10-25 23:00:00')]
IGI Missing timestamps: [Timestamp('2025-10-25 23:00:00')]
NSIT Missing timestamps: [Timestamp('2025-10-25 23:00:00')]
